### IMPORTS

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### SPARK AND ADLS CONFIGURATION

In [0]:
# Spark configuration for ADLS Gen2 OAuth authentication
# TODO: Move client_id and client_secret to Azure Key Vault for production security
# Currently hardcoded for development purposes only

try:
    spark.conf.set("fs.azure.account.auth.type.spotifydlstorage.dfs.core.windows.net", "OAuth")
    spark.conf.set("fs.azure.account.oauth.provider.type.spotifydlstorage.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set("fs.azure.account.oauth2.client.id.spotifydlstorage.dfs.core.windows.net", "<your-client-id>")
    spark.conf.set("fs.azure.account.oauth2.client.secret.spotifydlstorage.dfs.core.windows.net", "<your-client-secret>")
    spark.conf.set("fs.azure.account.oauth2.client.endpoint.spotifydlstorage.dfs.core.windows.net", "https://login.microsoftonline.com/<your-tenant-id>/oauth2/token")
    print("Spark config set successfully ✓")
except Exception as e:
    print(f"Failed to set Spark config: {e}")
    raise

#### SILVER PATH

In [0]:
# Configuration
SILVER_PATH = "abfss://silver@spotifydlstorage.dfs.core.windows.net/spotify_tracks/"

In [0]:
df = spark.read.format("csv")\
    .option('header', True)\
    .option('inferSchema', True)\
    .load("abfss://bronze@spotifydlstorage.dfs.core.windows.net/*/*")

In [0]:
df.printSchema()

In [0]:
display(df.limit(5))

In [0]:
df.show(5)

### CHECK DATA 

In [0]:
print(f"Total rows: {df.count()}")
print(f"Unique track_ids: {df.select('track_id').distinct().count()}")

In [0]:
null_counts = df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
null_counts.show()

In [0]:
df.select(
    min("popularity"), max("popularity"),
    min("duration_ms"), max("duration_ms"),
    min("tempo"), max("tempo")
).show()

### DROPPING USELESS COLUMN, DROPPING ROWS WITH NULL COLUMN, CASTING THE RIGHT DATA TYPE 

In [0]:
df_silver = df.drop("Unnamed: 0")
df_silver = df_silver.na.drop(subset=['artists', "album_name", "track_name"])
df_silver = df_silver.withColumn("popularity", col("popularity").cast("integer"))\
                     .withColumn("duration_ms", col("duration_ms").cast("integer"))\
                     .withColumn("key", col("key").cast("integer"))\
                     .withColumn("mode", col("mode").cast("integer"))\
                     .withColumn("danceability", col("danceability").cast("double"))\
                     .withColumn("energy", col("energy").cast("double"))\
                     .withColumn("loudness", col("loudness").cast("double"))\
                     .withColumn("speechiness", col("speechiness").cast("double"))\
                     .withColumn("acousticness", col("acousticness").cast("double"))\
                     .withColumn("liveness", col("liveness").cast("double"))\
                     .withColumn("valence", col("valence").cast("double"))\
                     .withColumn("duration_min", round(col("duration_ms") / 60000, 2))
df_silver = df_silver.filter(col("tempo") >= 0)\
                     .filter(col("explicit").isin(["True", "False"]))


#### DATA QUALITY ASSERTIONS

In [0]:
# Data quality assertions
try:
    assert df_silver.count() > 0, "Silver DataFrame is empty"
    assert df_silver.count() == 113865, "Unexpected row count in silver layer"
    print("All assertions passed ✓")
except AssertionError as e:
    print(f"Assertion failed: {e}")
    raise

In [0]:
df_silver.printSchema()

In [0]:
bronze = df.count()
silver = df_silver.count()
print(f"Bronze Count: {bronze}")
print(f"Silver Count: {silver}")

### CHECKING NULLS 

In [0]:
df_silver.select([count(when(col(c).isNull(),c)).alias(c) for c in df_silver.columns]).show()

### RANGE CHECK 

In [0]:
display(df_silver.select(
    min("popularity"), max("popularity"),
    min("duration_ms"), max("duration_ms"),
    min("tempo"), max("tempo"),
    min("danceability"), max("danceability"),
    min("energy"), max("energy"),
    min("speechiness"), max("speechiness"),
    min("acousticness"), max("acousticness"),
    min("liveness"), max("liveness"), 
    min("valence"), max("valence"),
    min("loudness"), max("loudness"),
    min("key"), max("key"),
    min("mode"), max("mode"),
    min("time_signature"), max("time_signature")
))

### INVESTIGATE ANOMALIES

In [0]:
# ANOMALY INVESTIGATION

# 1. Negative tempo - 1 corrupted row found (opera track with shifted columns)
# Action: Dropped in transformation via filter(col("tempo") >= 0)
df_silver.filter(col("tempo") < 0).show()

# 2. Corrupted explicit rows - 133 rows found with garbage values from CSV column shifting
# Other columns (popularity, danceability) also NULL in these rows - unrecoverable
# Action: Dropped in transformation via filter(col("explicit").isin(["True", "False"]))
df_silver.filter(~col("explicit").isin(["True", "False"])).select("track_name", "artists", "popularity", "danceability", "explicit").show(20)

# 3. time_signature = 0 - 163 rows found (ambient, white noise, spoken word tracks)
# Other columns are valid - Spotify returns 0 when no beat is detectable
# Action: Retained - these are legitimate Spotify tracks
df_silver.filter(col("time_signature") == 0).select("track_name", "artists", "popularity", "danceability", "time_signature").show(10)

# 4. loudness > 0 - 90 rows found (heavily mastered electronic tracks, slight clipping above 0 dB)
# Other columns are valid - known characteristic of electronic/dance music production
# Action: Retained - legitimate audio phenomenon, not corrupt data
df_silver.filter(col("loudness") > 0).select("track_name", "artists", "popularity", "danceability", "loudness").show(10)

In [0]:
print("Null drop count:", df.drop("Unnamed: 0").na.drop(subset=['artists', 'album_name', 'track_name']).count())
print("Tempo filter count:", df.drop("Unnamed: 0").filter(col("tempo") >= 0).count())

In [0]:
df_silver.count()

In [0]:
try:
    df_silver.write.format("delta")\
                   .mode("overwrite")\
                   .save(SILVER_PATH)
    print("Silver layer written successfully")
except Exception as e:
    print(f"Failed to write silver layer: {e}")
    raise